# Project 3 
Completed by: Zarmeen Hasan

## Load and preprocess the data

In [2]:
import pandas as pd 

In [3]:
# Reading in movies.dat
movies_cols = ["movie_id", "title", "genres"]
movies_df = pd.read_csv(
    "data/movies.dat",
    sep="::",
    engine="python",
    names=movies_cols,
    encoding="latin-1",
)

movies_df.head()

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
# Reading in the ratings.dat
ratings_cols = ["user_id", "movie_id", "rating", "timestamp"]
ratings_df = pd.read_csv(
    "data/ratings.dat",
    sep="::",
    engine="python",
    names=ratings_cols,
    encoding="latin-1",
)

ratings_df.head()

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [5]:
# Reading in users.dat
users_cols = ["user_id", "gender", "age", "occupation", "zip_code"]
users_df = pd.read_csv(
    "data/users.dat",
    sep="::",
    engine="python",
    names=users_cols,
    encoding="latin-1",
)

users_df.head()

,user_id,gender,age,occupation,zip_code
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [6]:
# Join the 3 dataframes into 1
ratings_movies = pd.merge(ratings_df, movies_df, on="movie_id")
df = pd.merge(ratings_movies, users_df, on="user_id")
df.head()

,user_id,movie_id,rating,timestamp,title,genres,gender,age,occupation,zip_code
0,1,1193,5,978300760,One Flew Over the Cuckoo's Nest (1975),Drama,F,1,10,48067
1,1,661,3,978302109,James and the Giant Peach (1996),Animation|Children's|Musical,F,1,10,48067
2,1,914,3,978301968,My Fair Lady (1964),Musical|Romance,F,1,10,48067
3,1,3408,4,978300275,Erin Brockovich (2000),Drama,F,1,10,48067
4,1,2355,5,978824291,"Bug's Life, A (1998)",Animation|Children's|Comedy,F,1,10,48067


In [7]:
# Check for any NAN values 
df.isnull().sum()

user_id       0
movie_id      0
rating        0
timestamp     0
title         0
genres        0
gender        0
age           0
occupation    0
zip_code      0
dtype: int64

In [8]:
# Check the number of unique user ids and movies ids to make sure they're what we expect 
print("Unique user IDs:", df["user_id"].nunique())
print("Unique movie IDs:", df["movie_id"].nunique())

Unique user IDs: 6040
Unique movie IDs: 3706


In [9]:
# Check how many movies were in movies.dat vs. ratings.dat. The number is probably different because some movies weren't rated
print("Movies in movies.dat:", movies_df['movie_id'].nunique())
print("Movies in ratings.dat:", ratings_df['movie_id'].nunique())
print("Movies after merge:", df['movie_id'].nunique())

Movies in movies.dat: 3883
Movies in ratings.dat: 3706
Movies after merge: 3706


In [10]:
# Convert `genres` column into a usable form. I am keeping this dataframe handy in case I need it for later. 
# For now, there's no need to merge it back into the full df: `df`.
from sklearn.preprocessing import MultiLabelBinarizer

df['genres_list'] = df['genres'].apply(lambda x: x.split('|'))
mlb = MultiLabelBinarizer()
genre_features = mlb.fit_transform(df['genres_list'])
genre_df = pd.DataFrame(genre_features, columns=mlb.classes_)
genre_df.head()

,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0
3,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
4,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0


## Problem A

In [11]:
# Create a popularity dataframe with rating counts and average rating per movie. 
popularity_df = (
    df.groupby(['movie_id', 'title'])
    .agg(num_ratings=('rating', 'count'),
         avg_rating=('rating', 'mean'))
    .reset_index()
)
popularity_df.head()

,movie_id,title,num_ratings,avg_rating
0,1,Toy Story (1995),2077,4.146846
1,2,Jumanji (1995),701,3.201141
2,3,Grumpier Old Men (1995),478,3.016736
3,4,Waiting to Exhale (1995),170,2.729412
4,5,Father of the Bride Part II (1995),296,3.006757


In [12]:
# I am defining the popularity of a movie by the highest average rating (with at least 1000 ratings)
print("Top 10 'most popular' movies:")
popularity_df[popularity_df['num_ratings'] >= 1000].sort_values('avg_rating', ascending=False).head(10)

Top 10 'most popular' movies:


,movie_id,title,num_ratings,avg_rating
309,318,"Shawshank Redemption, The (1994)",2227,4.554558
802,858,"Godfather, The (1972)",2223,4.524966
49,50,"Usual Suspects, The (1995)",1783,4.517106
513,527,Schindler's List (1993),2304,4.510417
1108,1198,Raiders of the Lost Ark (1981),2514,4.477725
843,904,Rear Window (1954),1050,4.476190
253,260,Star Wars: Episode IV - A New Hope (1977),2991,4.453694
713,750,Dr. Strangelove or: How I Learned to Stop Worr...,1367,4.449890
851,912,Casablanca (1942),1669,4.412822
2557,2762,"Sixth Sense, The (1999)",2459,4.406263


## Problem B 

### Part 1

In [13]:
import numpy as np

In [14]:
# Create the user-movie matrix R 
R = ratings_df.pivot(index="user_id", columns="movie_id", values="rating")
R.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# Normalize each row 
row_means = R.mean(axis=1, skipna=True)
R = R.sub(row_means, axis=0).fillna(0)

In [16]:
R.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
user_id,,,,,,,,,,,,,,,,,,,,,
1,0.811321,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.000000,0.0,0.0,0.0,0.0,-1.146465,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Part 2

In [17]:
# Use sparse math because doing a double for loop will take too long to run 
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

R_sparse = csr_matrix(R.values)
sim_matrix = cosine_similarity(R_sparse.T, dense_output=False)

In [18]:
# Convert similarity matrix to a dataframe
sim_df = pd.DataFrame(sim_matrix.toarray(), index=R.columns, columns=R.columns)

# Apply the (1+cos)/2 transformation 
sim_df = (1 + sim_df) / 2

In [19]:
# Count how many users rated both movies
binary_R = (~R.isna()).astype(int)
co_ratings = binary_R.T.dot(binary_R)

# Mask similarities where fewer than 3 shared ratings
sim_df[co_ratings < 3] = np.nan

In [20]:
sim_df.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
movie_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.474227,0.470569,0.478718,0.454097,0.531066,0.490987,0.471598,0.450709,0.505509,...,0.487664,0.487200,0.458473,0.478445,0.509439,0.515507,0.539883,0.505599,0.509909,0.518752
2,0.474227,1.000000,0.510341,0.494456,0.531128,0.475039,0.517661,0.498533,0.534299,0.516363,...,0.492896,0.498306,0.496375,0.518117,0.480145,0.514147,0.468532,0.486118,0.492839,0.486457
3,0.470569,0.510341,1.000000,0.527792,0.554713,0.485404,0.515214,0.512748,0.514612,0.519470,...,0.496304,0.498876,0.496222,0.515944,0.500489,0.505155,0.467769,0.501026,0.499735,0.491413
4,0.478718,0.494456,0.527792,1.000000,0.563312,0.471319,0.502506,0.508409,0.499291,0.484169,...,0.516171,0.493269,0.511059,0.526708,0.497542,0.493944,0.446060,0.501694,0.503797,0.495115
5,0.454097,0.531128,0.554713,0.563312,1.000000,0.477780,0.528325,0.506986,0.543323,0.500863,...,0.488702,0.500320,0.510144,0.513637,0.505909,0.514638,0.467124,0.500630,0.502846,0.482833


### Part 3

In [21]:
sim_top30 = sim_df.copy()
for movie in sim_df.columns:
    # Get similarities for this movie 
    sims = sim_df[movie].copy()
    sims[movie] = np.nan  # ensure self-similarity is not counted
    
    # Find top k similarity values (excluding NaN)
    topk_indices = sims.nlargest(30).index
    
    # Set all other values to NaN
    sim_top30.loc[~sim_top30.index.isin(topk_indices), movie] = np.nan

In [22]:
sim_top30.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
movie_id,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,0.554713,0.563312,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
# create a title to movie lookup dictionary
title_to_id = dict(zip(movies_df['title'], movies_df['movie_id']))

In [24]:
# Get the movie IDs 
target_titles = [
    "Toy Story (1995)",
    "GoldenEye (1995)",
    "Liar Liar (1997)",
    "Lost World: Jurassic Park, The (1997)", # for some reason movies.dat wrote the title of the movie like this
    "Sixth Sense, The (1999)"
]

target_ids = [title_to_id[t] for t in target_titles if t in title_to_id]
print(target_ids)

[1, 10, 1485, 1544, 2762]


In [25]:
# Extract the movies from the similarity matrix 
sim_target = sim_top30.loc[target_ids, target_ids]

# Map back to movie titles
sim_target.index = [movies_df.loc[movies_df['movie_id'] == i, 'title'].values[0] for i in sim_target.index]
sim_target.columns = [movies_df.loc[movies_df['movie_id'] == i, 'title'].values[0] for i in sim_target.columns]

sim_target


,Toy Story (1995),GoldenEye (1995),Liar Liar (1997),"Lost World: Jurassic Park, The (1997)","Sixth Sense, The (1999)"
Toy Story (1995),NaN,NaN,NaN,NaN,0.62578
GoldenEye (1995),NaN,NaN,0.544694,NaN,NaN
Liar Liar (1997),NaN,NaN,NaN,NaN,NaN
"Lost World: Jurassic Park, The (1997)",NaN,NaN,NaN,NaN,NaN
"Sixth Sense, The (1999)",0.62578,NaN,NaN,NaN,NaN


### Part 4

In [26]:
def _get_popularity_ranking(popularity_df):
    popular_movies = popularity_df[popularity_df['num_ratings'] >= 1000]
    popular_movies_sorted = popular_movies.sort_values('avg_rating', ascending=False)
    popularity_ranking = popular_movies_sorted['movie_id'].tolist()
    return popularity_ranking

In [27]:
def IBCF_recommend_top10(newuser):
    """
    newuser: pd.Series (index = movieId, values = 1-5 or NaN)
    Returns top 10 recommended movies as DataFrame
    """
    sim_use = sim_top30 

    rated = newuser.dropna()
    unrated = newuser[newuser.isna()]

    preds = {}
    for movie_id in unrated.index:
        if movie_id not in sim_use.index:
            preds[movie_id] = np.nan
            continue

        sims = sim_use.loc[movie_id, rated.index].dropna()
        if len(sims) < 3:
            preds[movie_id] = np.nan
            continue

        weights = sims.values
        ratings = rated.loc[sims.index].values
        denom = np.sum(np.abs(weights))
        preds[movie_id] = np.dot(weights, ratings) / denom if denom != 0 else np.nan

    preds = pd.Series(preds).dropna().sort_values(ascending=False)

    # If fewer than 10 predictions, fill with popularity
    recs = list(preds.index[:10])
    remaining = [m for m in _get_popularity_ranking(popularity_df)
                 if m not in rated.index and m not in recs]
    recs.extend(remaining[:10 - len(recs)])

    result = movies_df[movies_df['movie_id'].isin(recs)][['movie_id', 'title']].copy()
    result['predicted_rating'] = result['movie_id'].map(preds)
    result = result.sort_values('predicted_rating', ascending=False, na_position='last')

    return result.reset_index(drop=True)


### Part 5

In [28]:
# Test case 1
user_1500 = R.iloc[1500, :].copy()
user_1500[user_1500 == 0] = np.nan
recs_user1500 = IBCF_recommend_top10(user_1500)
display(recs_user1500)


,movie_id,title,predicted_rating
0,261,Little Women (1994),1.273063
1,1721,Titanic (1997),1.071321
2,3755,"Perfect Storm, The (2000)",1.020673
3,361,It Could Happen to You (1994),0.936990
4,1878,Woo (1998),0.935454
5,2300,"Producers, The (1968)",0.793643
6,2336,Elizabeth (1998),0.787689
7,900,"American in Paris, An (1951)",0.784921
8,517,Rising Sun (1993),0.781769
9,2125,Ever After: A Cinderella Story (1998),0.776732


In [32]:
# Test case 2: Create a new hypothetical user with several ratings

# Choose a few known movie titles to rate
rated_titles = [
    "Star Wars: Episode IV - A New Hope (1977)",
    "Independence Day (ID4) (1996)",
    "Jurassic Park (1993)",
    "Toy Story (1995)",
    "Matrix, The (1999)",
    "Titanic (1997)",
    "Forrest Gump (1994)",
]
rated_scores = [5, 4, 5, 4, 5, 3, 4]

rated_ids = [title_to_id[t] for t in rated_titles if t in title_to_id]

newuser = pd.Series(np.nan, index=R.columns)

for mid, score in zip(rated_ids, rated_scores):
    newuser[mid] = score

recs_hypothetical = IBCF_recommend_top10(newuser)
display(recs_hypothetical)


,movie_id,title,predicted_rating
0,1214,Alien (1979),5.000000
1,1200,Aliens (1986),5.000000
2,1240,"Terminator, The (1984)",5.000000
3,1210,Star Wars: Episode VI - Return of the Jedi (1983),4.768548
4,110,Braveheart (1995),4.748308
5,1197,"Princess Bride, The (1987)",4.679576
6,3578,Gladiator (2000),4.671454
7,1610,"Hunt for Red October, The (1990)",4.671201
8,1196,Star Wars: Episode V - The Empire Strikes Back...,4.630434
9,589,Terminator 2: Judgment Day (1991),4.621081
